In [0]:
# ==========================================
# SILVER SCORERS - SOURCE INSPECTION
# ==========================================

# Read the original Bronze Delta table
bronze_df = spark.table("workspace.default.bronze_scorers")

# Inspect the nested JSON structure
bronze_df.printSchema()

# Display the top-level columns
print("Columns:", bronze_df.columns)

# Count the number of players in the scorers array
from pyspark.sql.functions import col, size

bronze_df.select(
    size(col("scorers")).alias("player_count")
).show()

# Preview the original scorers data
display(bronze_df.select("scorers"))


In [0]:
from pyspark.sql.functions import col, explode, to_date

# ==========================================
# 1. READ BRONZE SCORERS
# ==========================================

bronze_df = spark.table("workspace.default.bronze_scorers")

# ==========================================
# 2. FLATTEN THE SCORERS ARRAY
# ==========================================

scorers_df = bronze_df.select(
    col("season.id").alias("season_id"),
    explode(col("scorers")).alias("scorer")
)


# ==========================================
# 3. SELECT PLAYER STATISTICS
# ==========================================

silver_df = scorers_df.select(

    # Season identifier
    col("season_id"),

    # Player details
    col("scorer.player.id").alias("player_id"),
    col("scorer.player.name").alias("player_name"),
    col("scorer.player.nationality").alias("nationality"),
    col("scorer.player.position").alias("position"),
    to_date(col("scorer.player.dateOfBirth")).alias("date_of_birth"),

    # Team details
    col("scorer.team.id").alias("team_id"),
    col("scorer.team.name").alias("team_name"),

    # Performance statistics
    col("scorer.playedMatches").alias("matches_played"),
    col("scorer.goals").alias("goals"),
    col("scorer.assists").alias("assists"),
    col("scorer.penalties").alias("penalties")
)

display(silver_df)


In [0]:
# ==========================================
# 4. DATA QUALITY CHECKS
# ==========================================

# Remove records without essential identifiers
silver_df = silver_df.dropna(
    subset=["season_id", "player_id", "team_id"]
)

# Remove duplicate player records within a season
silver_df = silver_df.dropDuplicates(
    ["season_id", "player_id"]
)

# Count the cleaned records
total_players = silver_df.count()

# Validate unique player IDs per season
unique_players = silver_df.select(
    "season_id", "player_id"
).distinct().count()

assert total_players == unique_players, "Duplicate player records found!"

# Validate goals are not negative
invalid_goals = silver_df.filter(
    col("goals") < 0
).count()

assert invalid_goals == 0, "Invalid goals detected!"

# Validate matches played are not negative
invalid_matches = silver_df.filter(
    col("matches_played") < 0
).count()

assert invalid_matches == 0, "Invalid matches played detected!"

print("Scorers validation passed!")
print("Total clean players:", total_players)

# ==========================================
# 5. SAVE SILVER SCORERS
# ==========================================

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_scorers")
)

print("Silver scorers table created successfully!")


In [0]:

%sql
SELECT
    player_name,
    team_name,
    matches_played,
    goals,
    assists,
    penalties
FROM workspace.default.silver_scorers
ORDER BY goals DESC, assists DESC;
